In [1]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI


In [2]:

documents = load_faq_data()
index = build_index(documents)


In [3]:
print(f"Loaded {len(documents)} documents")

Loaded 1350 documents


In [4]:

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)


In [5]:
answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join now. If you want a certificate, make sure you submit your project while submissions are still open.


# Agents


In [6]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—most likely, but it depends on the course’s enrollment rules and whether it’s still open.\n\nTo check, look for:\n- **Enrollment deadline / start date**\n- **Prerequisites**\n- **Seat availability**\n- **Waitlist option**\n- **Registration page or course admin contact**\n\nIf you want, send me the **course name or link**, and I can help you figure out whether you can still join and what to do next.'

In [7]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [9]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? discovered the course can I join"}', call_id='call_7tpIftRyck99vDmHaPq5DZ6Q', name='search', type='function_call', id='fc_08150a00190c1f6c006a3ff052ab4c819cb0e42aade5521d9e', namespace=None, status='completed')]

In [10]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [13]:
from pprint import pprint
pprint(result_json)

('[\n'
 '  {\n'
 '    "id": "74eb249bbf",\n'
 '    "course": "llm-zoomcamp",\n'
 '    "section": "General Course-Related Questions",\n'
 '    "question": "I just discovered the course. Can I still join?",\n'
 '    "answer": "Yes, but if you want to receive a certificate, you need to '
 'submit your project while we\\u2019re still accepting submissions."\n'
 '  },\n'
 '  {\n'
 '    "id": "69d122f12e",\n'
 '    "course": "llm-zoomcamp",\n'
 '    "section": "General Course-Related Questions",\n'
 '    "question": "Certificate: Can I follow the course in a self-paced mode '
 'and get a certificate?",\n'
 '    "answer": "No, you can only get a certificate if you finish the course '
 'with a \\"live\\" cohort.\\n\\nWe don\'t award certificates for the '
 'self-paced mode. The reason is you need to peer-review 3 capstone(s) after '
 'submitting your project.\\n\\nYou can only peer-review projects at the time '
 'the course is running; after the form is closed and the peer-review list is '
 'c

In [14]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [15]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open.'

In [16]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(845, 33)

In [17]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176
